# Demo: predict an emotion and see where the model looks (#15)

Give it a face photo and it prints the predicted emotion with class probabilities, and draws a Grad-CAM heatmap of
the image regions that pushed the model toward that emotion.

**Before you run it**
1. The repo is cloned and `timm`, `grad-cam`, `torch`, `torchvision` and `pillow` are installed
   (on Kaggle or Colab: `!pip install -q timm grad-cam`).
2. The checkpoint is in `results/vit/best.pt`. Checkpoints are not in git; download it from the team Drive.
   The matching `config.yaml` is already in the repo.
3. Set `IMAGE_PATH` below to a photo of one face.

The model was trained on small, centred, greyscale FER-2013 faces, so crop the photo to the face first. There is no
face detector in this notebook. Colour photos are fine, they are converted to greyscale before the model sees them.

In [ ]:
# IMPORTS + PATH SETUP: ROOT is the repo folder (one level up if the notebook was opened from notebooks/); adding it to sys.path
# is what lets "from src.common import ..." work. The demo reuses the project's own load_run / build_transforms / Grad-CAM layer
# choice, so it behaves exactly like the evaluation scripts.
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():  # opened from notebooks/
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.common import get_device, load_run
from src.datasets import CLASSES, build_transforms
from src.models import gradcam_target_layer

device = get_device()
print("device:", device)

## Settings

In [ ]:
# SETTINGS: RUNS = which trained models to show; IMAGE_PATH = your face photo (None = a random FER-2013 test image);
# TARGET_EMOTION = which emotion's heat map to draw (None = the emotion the model predicts).
# runs to show, one row each. Needs results/<run>/best.pt. The final ViT is "vit" (ViT-B);
# with all three checkpoints you can compare: ["cnn_best", "resnet50", "vit"]
RUNS = ["vit"]

# a photo of one face. If None, a random FER-2013 test image is used when data/fer2013 exists
IMAGE_PATH = None

# e.g. "sad" to see where the model looks for that emotion; None = the emotion it predicts
TARGET_EMOTION = None

In [ ]:
# INPUT: nothing (uses IMAGE_PATH).  RETURNS: a PIL image.  FAILS WITH FileNotFoundError if there is no photo and no FER-2013 folder.
def load_image():
    if IMAGE_PATH:
        return Image.open(IMAGE_PATH).convert("RGB")
    files = sorted((ROOT / "data" / "fer2013" / "test").glob("*/*.*"))
    if not files:
        raise FileNotFoundError("Set IMAGE_PATH to a face photo (or put FER-2013 under data/fer2013).")
    return Image.open(files[np.random.default_rng().integers(len(files))]).convert("RGB")


# Same idea as in src/explain.py: a ViT gives a flat list of tokens, so drop the CLS token and fold the patch tokens back into a square map.
def vit_reshape(t, size):
    # drop the CLS token, [B, N, C] -> [B, C, H, W]
    return t[:, 1:, :].reshape(t.size(0), size, size, t.size(2)).permute(0, 3, 1, 2)


# Undo the normalisation so the original-looking face can be drawn under the heat map.
def to_display(x, channels):
    # undo the normalisation so the face can sit under the heatmap
    img = x.permute(1, 2, 0).numpy()
    if channels == 3:
        img = img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
    else:
        img = np.repeat(img * 0.5 + 0.5, 3, axis=2)
    return np.clip(img, 0, 1).astype(np.float32)


# INPUT: a run name and a PIL image.  RETURNS: (class probabilities, predicted index, target index, heat-map overlay image).
# Steps: load saved model -> apply the same preprocessing as training -> softmax -> Grad-CAM for the chosen emotion.
# IF best.pt IS MISSING: it raises FileNotFoundError telling you to download the checkpoint from the team Drive.
def explain(run, img):
    run_dir = ROOT / "results" / run
    if not (run_dir / "best.pt").exists():
        raise FileNotFoundError(f"{run_dir / 'best.pt'} is missing. Download it from the team Drive.")
    model, cfg = load_run(run_dir, device)
    name, d = cfg["model"]["name"], cfg["data"]

    x = build_transforms(d["img_size"], d["channels"], train=False)(img).unsqueeze(0).to(device)
    with torch.no_grad():
        probs = torch.softmax(model(x), dim=1)[0].cpu().numpy()
    pred = int(probs.argmax())
    target = CLASSES.index(TARGET_EMOTION) if TARGET_EMOTION else pred

    size = d["img_size"] // 16
    cam = GradCAM(model=model, target_layers=[gradcam_target_layer(model, name)],
                  reshape_transform=(lambda t: vit_reshape(t, size)) if name.startswith("vit") else None)
    heat = cam(input_tensor=x, targets=[ClassifierOutputTarget(target)])[0]
    overlay = show_cam_on_image(to_display(x[0].cpu(), d["channels"]), heat, use_rgb=True)
    return probs, pred, target, overlay


# INPUT: the input image and the results dict.  RETURNS: nothing; draws input, heat map and probability bars for each run.
def show(img, results):
    fig, axes = plt.subplots(len(results), 3, figsize=(10, 3.2 * len(results)), squeeze=False)
    for row, (run, (probs, pred, target, overlay)) in zip(axes, results.items()):
        row[0].imshow(img)
        row[0].set_title("input")
        row[1].imshow(overlay)
        row[1].set_title(f"{run}: heatmap for '{CLASSES[target]}'")
        colors = ["#eb6834" if i == pred else "#9aa5b1" for i in range(len(CLASSES))]
        row[2].barh(CLASSES, probs, color=colors)
        row[2].invert_yaxis()
        row[2].set_xlim(0, 1)
        row[2].set_title(f"predicts {CLASSES[pred]} ({probs[pred]:.0%})")
        for ax in row[:2]:
            ax.axis("off")
        row[2].spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    plt.show()

## Run it

In [ ]:
# RUN IT: load one image, explain it with every model in RUNS, then draw the result.
img = load_image()
results = {run: explain(run, img) for run in RUNS}
show(img, results)

## How to read it

- The bars are the model's probabilities. They add up to 1 but are not calibrated, so 90% does not mean it is right
  nine times in ten.
- The heatmap is Grad-CAM for the chosen emotion: warm areas are the image regions that raised that class score. It
  shows where the model looked, not why. In our ResNet-50 runs the heat sat mostly on the mouth and nose.
- Treat the output as a demo of the models, not as a judgement about a person. The final ViT-B scored 70.7% on
  FER-2013 test but 64.7% on RAF-DB, and its accuracy differed between gender, age and race groups there (see
  `results/vit_tuning.md` and the bias audit in the report). Photos unlike FER-2013 will do worse.